# 07 - Build a Transformer block

This notebook combines multi-head causal self-attention with the other components needed in a decoder block: layer normalization, residual connections, and a position-wise feed-forward network. The example uses random tensors and does not load Darija data or train model weights.

## 1. Set the model dimensions

Each token position is represented by a vector of width `d_model`. Attention and the feed-forward network both return the same width, which allows their outputs to be added back through residual connections. The intermediate feed-forward width is commonly larger; here it is four times `d_model`.

In [ ]:
import math
import torch
from torch import nn

torch.manual_seed(123)

batch_size = 2
sequence_length = 4
d_model = 8
num_heads = 2
context_length = 16
dropout_rate = 0.0

x = torch.randn(batch_size, sequence_length, d_model)
print("input:", tuple(x.shape))

## 2. Use causal multi-head self-attention

This notebook is self-contained, so the attention module is defined here as well. It projects each position into queries, keys, and values, splits the features into heads, blocks attention to future positions, and concatenates the head outputs. The model width must be divisible by the number of heads.

In [ ]:
class MultiHeadCausalSelfAttention(nn.Module):
    def __init__(self, d_model, context_length, num_heads, dropout=0.0):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")

        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.d_model = d_model
        self.W_query = nn.Linear(d_model, d_model, bias=False)
        self.W_key = nn.Linear(d_model, d_model, bias=False)
        self.W_value = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        mask = torch.triu(torch.ones(context_length, context_length, dtype=torch.bool), diagonal=1)
        self.register_buffer("future_mask", mask)

    def forward(self, x):
        batch, steps, _ = x.shape
        if steps > self.future_mask.shape[0]:
            raise ValueError("sequence length exceeds context_length")
        shape = (batch, steps, self.num_heads, self.head_dim)
        q = self.W_query(x).view(shape).transpose(1, 2)
        k = self.W_key(x).view(shape).transpose(1, 2)
        v = self.W_value(x).view(shape).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = self.future_mask[:steps, :steps]
        scores = scores.masked_fill(mask, -torch.inf)
        weights = self.dropout(torch.softmax(scores, dim=-1))
        context = weights @ v
        context = context.transpose(1, 2).contiguous().view(batch, steps, self.d_model)
        return self.out_proj(context)

## 3. Add a feed-forward network

Attention mixes information across token positions. The feed-forward network then transforms each position independently: it expands the vector, applies the GELU activation, and projects it back to `d_model`. The same weights are applied at every position.

In [ ]:
feed_forward = nn.Sequential(
    nn.Linear(d_model, 4 * d_model),
    nn.GELU(),
    nn.Linear(4 * d_model, d_model),
)

ff_output = feed_forward(x)
print("feed-forward output:", tuple(ff_output.shape))

## 4. Normalize and preserve residual paths

This block uses pre-layer normalization: normalize the input to each sublayer, then add the sublayer result to its original input. The residual path keeps information flowing through the block and gives gradients a direct route backward. Dropout can regularize each residual branch; it is set to zero in this example.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, context_length, num_heads, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attention = MultiHeadCausalSelfAttention(
            d_model=d_model, context_length=context_length, num_heads=num_heads, dropout=dropout
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout(self.attention(self.norm1(x)))
        x = x + self.dropout(self.feed_forward(self.norm2(x)))
        return x

## 5. Run a forward and backward pass

The output keeps the same `(batch, sequence, model_width)` shape as the input. A small scalar objective lets us check that gradients reach the input and the block's trainable parameters. This is a computation check only; no optimizer step or model training is performed.

In [ ]:
block = TransformerBlock(
    d_model=d_model, context_length=context_length, num_heads=num_heads, dropout=dropout_rate
)
x = x.requires_grad_()
block_output = block(x)
loss = block_output.square().mean()
loss.backward()

assert block_output.shape == (batch_size, sequence_length, d_model)
assert x.grad is not None and torch.isfinite(x.grad).all()
assert block.attention.W_query.weight.grad is not None
print("block output:", tuple(block_output.shape))
print("input gradient shape:", tuple(x.grad.shape))
print("gradient reaches query projection:", block.attention.W_query.weight.grad is not None)

## Key idea

A Transformer block has two sublayers: causal self-attention and a position-wise feed-forward network. Each sits on a residual path and receives a normalized input. Stacking these blocks creates the decoder body; the next step is to add token and positional embeddings and assemble the language model.